# Работа с SQL базой в pandas

Для работы с базой через pandas требуется предварительно настроить движок на основе библиотеки SQLAlchemy.

В этом задании также будет рассмотрена работа с mysql базой sakila.

In [1]:
import pandas as pd
import sqlalchemy as sa

user='admin'
password='admin'
host='MYSQL'
port=3306
database='sakila'

sql_dialect="mysql"
sql_driver="mysqlconnector"

try:
    engine = sa.create_engine(f"{sql_dialect}+{sql_driver}://{user}:{password}@{host}:{port}/{database}")    
    with engine.connect() as cnx:
        result = cnx.execute(sa.text("SELECT 1"))
        print("SQLAlchemy engine works properly")
except Exception as e:
    print(f"Error connecting to database: {e}")

SQLAlchemy engine works properly


Когда движок настроен можно загружать из базы таблицы целиком в виде DataFrame

In [2]:
table_name = "city"
df = pd.read_sql_table(table_name, engine)
df.head()

,city_id,city,country_id,last_update
0,1,A Coruña (La Coruña),87,2006-02-15 04:45:25
1,2,Abha,82,2006-02-15 04:45:25
2,3,Abu Dhabi,101,2006-02-15 04:45:25
3,4,Acuña,60,2006-02-15 04:45:25
4,5,Adana,97,2006-02-15 04:45:25


Также можно выполнять SQL запросы и загружать их результаты в виде DataFrame

In [3]:
qu = "SELECT * FROM city"
df = pd.read_sql_query(qu, engine)
df.head()

,city_id,city,country_id,last_update
0,1,A Coruña (La Coruña),87,2006-02-15 04:45:25
1,2,Abha,82,2006-02-15 04:45:25
2,3,Abu Dhabi,101,2006-02-15 04:45:25
3,4,Acuña,60,2006-02-15 04:45:25
4,5,Adana,97,2006-02-15 04:45:25


## Задание 1

Постройте SQL запрос, который возвращает список названий фильмов и языков, на которых он сняты. Далее выполните этот запрос при помощи pandas и исползуя полученный набора данных подсчитайте, сколько фильмов на каждом из языков имеются в базе.

In [12]:
# Ваш код:
qu = """SELECT f.title, l.name
    FROM film as f
    JOIN language as l ON f.language_id = l.language_id
    """
df = pd.read_sql_query(qu, engine)
df.head()

df.groupby("name")["title"].count()

name
English    1000
Name: title, dtype: int64

## Задание 2

Загрузите таблицы 'film_category' и 'category' в pandas и удалите из полученных наборов данных столбцы 'last_update'. Изучите документацию метода `.merge()` и выполните объединение считанных наборов данных. Используя полученный набор данных подсчитайте количество фильмов каждой категории.

In [5]:
# Ваш код:
film_category_df = pd.read_sql_query("SELECT * FROM film_category", engine).drop(columns = "last_update")
category_df = pd.read_sql_query("SELECT * FROM category", engine).drop(columns = "last_update")
film_category_df.head()

,film_id,category_id
0,1,6
1,2,11
2,3,6
3,4,11
4,5,8


In [6]:
category_df.head()

,category_id,name
0,1,Action
1,2,Animation
2,3,Children
3,4,Classics
4,5,Comedy


In [7]:
megre_category = pd.merge(film_category_df, category_df, on="category_id")
megre_category.head()

,film_id,category_id,name
0,1,6,Documentary
1,2,11,Horror
2,3,6,Documentary
3,4,11,Horror
4,5,8,Family


In [8]:
megre_category.groupby("name")["film_id"].count()

name
Action         64
Animation      66
Children       60
Classics       57
Comedy         58
Documentary    68
Drama          62
Family         69
Foreign        73
Games          61
Horror         56
Music          51
New            63
Sci-Fi         61
Sports         74
Travel         57
Name: film_id, dtype: int64

## Задание 3

Постройте запрос, который возвращает два столбца — 'person' (объединённые имя и фамилия клиента) и 'title' (название фильма, который клиент брал в прокате). 
Загрузите результат этого запроса в виде набора данных pandas. Далее, на основе него, в pandas постройте новый набор данных в котором в первом столбце стояли бы имя и фамилия клиента, а во втором — текстовая строка, в которой перечислены все фильмы, которые он брал. Для этого, вероятно, потребуется написать собственную функцию агрегации. Предусмотрите удаление повторов — название каждого фильма должно появляться только один раз. Фильмы должны быть отсортированы по алфавиту.

In [9]:
qu  = """SELECT CONCAT (c.first_name,' ', c.last_name) AS person, f.title
    FROM rental AS r
    JOIN customer AS c ON r.customer_id = c.customer_id
    JOIN inventory AS i ON r.inventory_id = i.inventory_id
    JOIN film AS f ON i.film_id = f.film_id
    """

film_df = pd.read_sql_query(qu, engine)
film_df.head()

,person,title
0,CHARLOTTE HUNTER,BLANKET BEVERLY
1,TOMMY COLLAZO,FREAKY POCUS
2,MANUEL MURRELL,GRADUATE LORD
3,ANDREW PURDY,LOVE SUICIDES
4,DELORES HANSEN,IDOLS SNATCHERS


In [13]:
def merge_film(films):
    # удаляем повторы и сортируем по алфавиту
    unique_sorted = sorted(set(films.values))
    return "; ".join(unique_sorted)

merge_film_df = pd.DataFrame(film_df.groupby("person")["title"].apply(merge_film)).reset_index()

merge_film_df.head()

,person,title
0,AARON SELBY,ARACHNOPHOBIA ROLLERCOASTER; BEAUTY GREASE; CO...
1,ADAM GOOCH,ALADDIN CALENDAR; BAREFOOT MANCHURIAN; COMMAND...
2,ADRIAN CLARY,BOOGIE AMELIE; BOUND CHEAPER; CHICAGO NORTH; D...
3,AGNES BISHOP,ANACONDA CONFESSIONS; ATLANTIS CAUSE; CAMELOT ...
4,ALAN KAHN,AGENT TRUMAN; ALIEN CENTER; BANGER PINOCCHIO; ...


## Задание 4

Сначала посредством SQL и pandas получите набор данных, который демонстрирует результативность сотрудников проката. Затем выберите самого результативного и получите набор данных, в котором в качестве индекса задано название фильма и имеется единственная колонка с описанием фильма. Вероятно, для этого потребуется использовать метод `.set_index()`.

In [19]:
# Ваш код:
qu_perfomance = """SELECT r.rental_id, r.staff_id, s.first_name, s.last_name
    FROM rental AS r
    JOIN staff AS s ON r.staff_id = s.staff_id
    """

perfomace_df = pd.read_sql_query(qu_perfomance, engine)
perfomace_df.head()

,rental_id,staff_id,first_name,last_name
0,1,1,Mike,Hillyer
1,2,1,Mike,Hillyer
2,3,1,Mike,Hillyer
3,5,1,Mike,Hillyer
4,6,1,Mike,Hillyer


In [35]:
staff_counts = perfomace_df.groupby(["staff_id", "first_name", "last_name"])["rental_id"].count()
staff_counts

staff_id  first_name  last_name
1         Mike        Hillyer      8040
2         Jon         Stephens     8004
Name: rental_id, dtype: int64

In [37]:
top_staff_id = staff_counts.idxmax()[0]
top_staff_id

np.int64(1)

In [38]:
qu = f"""
SELECT f.title, f.description
FROM rental AS r
JOIN staff AS s ON r.staff_id = s.staff_id
JOIN inventory AS i ON r.inventory_id = i.inventory_id
JOIN film AS f ON i.film_id = f.film_id
WHERE s.staff_id = {top_staff_id}
"""

result_df = pd.read_sql_query(qu, engine).set_index("title")
result_df.head()

,description
title,
BLANKET BEVERLY,A Emotional Documentary of a Student And a Gir...
FREAKY POCUS,A Fast-Paced Documentary of a Pastry Chef And ...
GRADUATE LORD,A Lacklusture Epistle of a Girl And a A Shark ...
IDOLS SNATCHERS,A Insightful Drama of a Car And a Composer who...
MYSTIC TRUMAN,A Epic Yarn of a Teacher And a Hunter who must...
